# Kaggle - Régression

Notebook nettoyé et corrigé à partir du notebook fourni.

**Objectifs :**
- charger et vérifier les données ;
- corriger le problème d'alignement de `y` ;
- analyser les valeurs manquantes ;
- utiliser `GroupKFold` par patient pour éviter le data leakage ;
- comparer un modèle Dummy à un modèle de régression ;
- générer un fichier `submission.csv` propre.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingRegressor


## 1. Chargement des données

In [ ]:
X = pd.read_csv("../data/X_train.csv")
y_raw = pd.read_csv("../data/y_train.csv")
X_test = pd.read_csv("../data/X_test.csv")

print("X_train :", X.shape)
print("y_train :", y_raw.shape)
print("X_test  :", X_test.shape)


## 2. Construction correcte de la target

Le fichier `y_train.csv` contient `Index` et `target`. Il ne faut **pas** donner la colonne `Index` au modèle comme deuxième target.

C'était la cause du MAE artificiellement énorme (~6900) observé dans le notebook initial : `DummyRegressor` prédisait à la fois `Index` et `target`.

In [ ]:
# Vérification des colonnes attendues
assert "Index" in X.columns
assert "Index" in y_raw.columns
assert "target" in y_raw.columns
assert "Index" in X_test.columns

# Vérifier que les Index correspondent exactement
assert X["Index"].equals(y_raw["Index"]), "X_train et y_train ne sont pas alignés sur Index."

# La vraie target est une seule série
y = y_raw["target"].copy()

print("Target :")
print(y.describe())
print("\nExemples :")
print(y.head())


## 3. Vérification des patients et des visites

In [ ]:
PATIENT_ID = "patient_id"

print("Nombre de patients :", X[PATIENT_ID].nunique())
print("Nombre de visites :", len(X))
print("\nNombre de visites par patient :")
print(X.groupby(PATIENT_ID).size().describe())


## 4. Valeurs manquantes

In [ ]:
missing_count = X.isna().sum()
missing_percentage = X.isna().mean() * 100

missing_table = pd.DataFrame({
    "Nombre de valeurs manquantes": missing_count,
    "Pourcentage (%)": missing_percentage.round(2)
})

missing_table = missing_table[missing_table["Nombre de valeurs manquantes"] > 0]
missing_table = missing_table.sort_values(
    by="Nombre de valeurs manquantes", ascending=False
)

display(missing_table)


## 5. Préparation des features

`Index` est un identifiant de ligne et `patient_id` est utilisé comme groupe de validation. On ne les donne pas directement au modèle.

Les valeurs manquantes sont imputées **dans le pipeline**, donc uniquement à partir des données d'entraînement de chaque fold.

In [ ]:
DROP_COLUMNS = ["Index", "patient_id"]

X_model = X.drop(columns=DROP_COLUMNS).copy()
X_test_model = X_test.drop(columns=DROP_COLUMNS).copy()

categorical_features = X_model.select_dtypes(include=["object", "string", "category"]).columns.tolist()
numeric_features = X_model.select_dtypes(include=["number"]).columns.tolist()

print("Variables numériques :", numeric_features)
print("Variables catégorielles :", categorical_features)


## 6. Baseline correcte : DummyRegressor

In [ ]:
groups = X[PATIENT_ID]
cv = GroupKFold(n_splits=5)

dummy_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_model, y, groups), start=1):
    X_train_fold = X_model.iloc[train_idx]
    X_val_fold = X_model.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    model = DummyRegressor(strategy="mean")
    model.fit(np.zeros((len(X_train_fold), 1)), y_train_fold)
    predictions = model.predict(np.zeros((len(X_val_fold), 1)))

    mae = mean_absolute_error(y_val_fold, predictions)
    dummy_scores.append(mae)
    print(f"Fold {fold}: MAE = {mae:.4f}")

print(f"\nDummy Mean MAE: {np.mean(dummy_scores):.4f}")
print(f"Dummy Std MAE : {np.std(dummy_scores):.4f}")


## 7. Modèle de régression

On utilise ici un `HistGradientBoostingRegressor` avec un preprocessing propre. Les variables catégorielles sont encodées en one-hot et les variables numériques sont imputées par leur médiane.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

model = HistGradientBoostingRegressor(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])


In [ ]:
scores = []
rmse_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_model, y, groups), start=1):
    X_train_fold = X_model.iloc[train_idx]
    X_val_fold = X_model.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    pipeline.fit(X_train_fold, y_train_fold)
    predictions = pipeline.predict(X_val_fold)

    mae = mean_absolute_error(y_val_fold, predictions)
    rmse = root_mean_squared_error(y_val_fold, predictions)

    scores.append(mae)
    rmse_scores.append(rmse)

    print(f"Fold {fold}: MAE = {mae:.4f} | RMSE = {rmse:.4f}")

print(f"\nMean MAE: {np.mean(scores):.4f}")
print(f"Std MAE : {np.std(scores):.4f}")
print(f"Mean RMSE: {np.mean(rmse_scores):.4f}")


## 8. Visualisation des prédictions du dernier fold

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_val_fold, predictions, alpha=0.25)

min_value = min(y_val_fold.min(), predictions.min())
max_value = max(y_val_fold.max(), predictions.max())
plt.plot([min_value, max_value], [min_value, max_value], linestyle="--")

plt.xlabel("True target")
plt.ylabel("Predicted target")
plt.title("True vs Predicted - dernier fold")
plt.show()


## 9. Entraînement final sur toutes les données

In [ ]:
pipeline.fit(X_model, y)
test_predictions = pipeline.predict(X_test_model)

print("Nombre de prédictions :", len(test_predictions))
print("Premières prédictions :", test_predictions[:10])


## 10. Création de la soumission Kaggle

In [ ]:
submission = pd.DataFrame({
    "Index": X_test["Index"],
    "target": test_predictions
})

assert len(submission) == len(X_test)
assert submission["Index"].equals(X_test["Index"])
assert submission["target"].notna().all()

display(submission.head())
print("Shape :", submission.shape)
print("Valeurs manquantes :")
print(submission.isna().sum())


In [ ]:
from pathlib import Path

Path("../submissions").mkdir(exist_ok=True)

submission_path = "../submissions/submission_hist_gradient_boosting.csv"
submission.to_csv(submission_path, index=False)

print(f"Fichier sauvegardé : {submission_path}")


## 11. Prochaines pistes d'amélioration

Avant de faire du tuning complexe, les pistes prioritaires sont :

1. Vérifier la définition exacte de la target et sa distribution.
2. Tester différentes stratégies d'imputation, notamment pour les variables très incomplètes.
3. Étudier les informations temporelles des visites par patient.
4. Créer des features d'évolution entre visites, uniquement avec les informations disponibles au moment de la prédiction.
5. Tester d'autres modèles : CatBoost, LightGBM/XGBoost si disponibles, ExtraTrees et Random Forest.
6. Faire du tuning avec le même `GroupKFold`.
7. Comparer systématiquement chaque amélioration à la baseline.

**Attention :** ne pas utiliser une information future du patient ou la target pour créer une feature, car cela provoquerait du data leakage.